# Auto Table

Running this notebook will generate a table for each specific star in their auto_plots folder that includes useful info such as Id, ra, dec, field, etc. 

**Recommedned to run auto_lomb for this to work best**

## Importing Libraries and Loading Data

In [ ]:
#import libraries and set up tap service
import matplotlib.pyplot as plt
import numpy as np
from astropy.table import Table
import pyvo
import astropy.units as u
from lsst.rsp import get_tap_service
import pandas as pd
from astropy.timeseries import LombScargle, LombScargleMultiband
pd.set_option('display.max_columns', None)

from lsst.utils.plotting import (get_multiband_plot_colors,
                                 get_multiband_plot_symbols)

from lsst.daf.butler import Butler
from lsst.utils.plotting import (
    get_multiband_plot_colors,
    get_multiband_plot_symbols,
    get_multiband_plot_linestyles
)

service = get_tap_service("tap")
assert service is not None

import seaborn as sns
my_pal = sns.color_palette("colorblind")

import warnings
warnings.filterwarnings('error')

In [ ]:
Ex_diaObj_df = pd.read_csv("data/Extra_diaObj_df.csv",index_col=0)
forcedDiaObj_df = pd.read_csv("data/diaForced_df.csv",index_col=0)

grouped_forcedDia = forcedDiaObj_df.groupby("diaObjectId")

filter_names = ['u', 'g', 'r', 'i', 'z', 'y']
filter_colors = get_multiband_plot_colors()
filter_symbols = get_multiband_plot_symbols()
filter_linestyles = get_multiband_plot_linestyles()

## Defining Functions to Save Tables

In [ ]:
def save_row_as_fig(row,lomb_df="",path=""):

    try:
        lomb_p_num = lomb_df[lomb_df["diaObjectId"] == row['diaObjectId']]["lomb_p1"].values[0]
        lomb_p = f"{lomb_p_num:.4f}"
    except:
        lomb_p = "N/A"

    RRL_class = row['DR3_best_classification']

    if RRL_class == "RRab":
        info_table = [
            ['diaObjectId', 'ObjectId', "DR3_source_id"],
            [row['diaObjectId'],row['ObjectId'],row['DR3_source_id']],
            ['Field','RA, Dec (deg)',"Classification"],
            [row['DR3_field_loc_center'],f"{row['ra']:.4f}, {row['dec']:.4f}",row['DR3_best_classification']],
            ['Mean r mag',"G mean mag",'Total DiaSources'],
            [f"{row['filt_mean_r_mag']:.4f}",f"{row['DR3_phot_g_mean_mag']:.4f}",row['nDiaSources']],
            ['GAIA ps Period (d)',"LombScargle Period","Total Good Sources"],
            [f"{row['DR3_pf']:.4f}",lomb_p,row["nGoodSources"]],
            ['nGoodu','nGoodg','nGoodr'],
            [row['nGoodu'],row['nGoodg'],row['nGoodr']],
            ['nGoodi','nGoody','nGoodz'],
            [row['nGoodi'],row['nGoody'],row['nGoodz']]
        ]
    elif RRL_class == "RRc":
        info_table = [
                ['diaObjectId', 'ObjectId', "DR3_source_id"],
                [row['diaObjectId'],row['ObjectId'],row['DR3_source_id']],
                ['Field','RA, Dec (deg)',"Classification"],
                [row['DR3_field_loc_center'],f"{row['ra']:.4f}, {row['dec']:.4f}",row['DR3_best_classification']],
                ['Mean r mag',"G mean mag",'Total DiaSources'],
                [f"{row['filt_mean_r_mag']:.4f}",f"{row['DR3_phot_g_mean_mag']:.4f}",row['nDiaSources']],
                ['GAIA p1_0 Period (d)',"LombScargle Period","Total Good Sources"],
                [f"{row['DR3_p1_o']:.4f}",lomb_p,row["nGoodSources"]],
                ['nGoodu','nGoodg','nGoodr'],
                [row['nGoodu'],row['nGoodg'],row['nGoodr']],
                ['nGoodi','nGoody','nGoodz'],
                [row['nGoodi'],row['nGoody'],row['nGoodz']]
            ]
    else:
        info_table = [
            ['diaObjectId', 'ObjectId', "DR3_source_id"],
            [row['diaObjectId'],row['ObjectId'],row['DR3_source_id']],
            ['Field','RA, Dec (deg)',"Classification"],
            [row['DR3_field_loc_center'],f"{row['ra']:.4f}, {row['dec']:.4f}",row['DR3_best_classification']],
            ['Mean r mag',"G mean mag",'Total DiaSources'],
            [f"{row['filt_mean_r_mag']:.4f}",f"{row['DR3_phot_g_mean_mag']:.4f}",row['nDiaSources']],
            ['GAIA p1_o/pf Period (d)',"LombScargle Period","Total Good Sources"],
            [f"{row['DR3_p1_o']:.4f}/{row['DR3_pf']:.4f}",lomb_p,row["nGoodSources"]],
            ['nGoodu','nGoodg','nGoodr'],
            [row['nGoodu'],row['nGoodg'],row['nGoodr']],
            ['nGoodi','nGoody','nGoodz'],
            [row['nGoodi'],row['nGoody'],row['nGoodz']]
        ]
        
        
        
    
    # Create figure and axis
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.axis('tight')
    ax.axis('off')
    
    # Create the table
    table = ax.table(cellText=info_table, loc='center', cellLoc='left')
    
    # Styling: Set font size and scale the table
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.2, 1.8)

    
    for j in range(0,len(info_table),2):
        for i in range(3):
            cell = table[j,i]
            cell.set_facecolor('rebeccapurple')
            cell.set_text_props(color='white', weight='bold')
            cell.get_text().set_weight('bold')
    
    for j in range(1,len(info_table),2):
        for i in range(3):
            cell = table[j,i]
            cell.set_facecolor('w')
            cell.set_text_props(color='k')

    plt.title(f"Stellar Information on {row['diaObjectId']}", fontsize=20, weight='bold',y=.9)
    plt.tight_layout()
    if path:
        plt.savefig(f"{path}/info_table.png",dpi=330,bbox_inches="tight")
    else:
        plt.show()
    plt.close()
    pass

In [ ]:
def filter_table(row,lomb_df="",path=""):
    print(f"Creating Table plot for {row["diaObjectId"]}...")
    
    wanted_columns =["diaObjectId","DR3_source_id","ObjectId","ra","dec",
                     "DR3_best_classification","DR3_field_loc_center",
                     "filt_mean_r_mag","DR3_phot_g_mean_mag",
                     "nDiaSources","nGoodSources",
                     "nGoodu","nGoodg","nGoodr","nGoodi","nGoodz","nGoody",
                     "query"]

    #may want to use DR3_phot_g_mean_flux and DR3_phot_g_mean_flux_error instead
    if row["DR3_best_classification"] == "RRab" or row["DR3_best_classification"] == "RRd":
        wanted_columns.append("DR3_pf")
        wanted_columns.append("DR3_pf_error")
    if row["DR3_best_classification"] == "RRc" or row["DR3_best_classification"] == "RRd":
        wanted_columns.append("DR3_p1_o")
        wanted_columns.append("DR3_p1_o_error")
        

    if path:
        # Creating New Folder for current star
        info_path = os.path.join(path,f"{row["diaObjectId"]}")
        
        if not os.path.isdir(info_path):
            os.mkdir(info_path)
    else:
        info_path = ""


    save_row_as_fig(row[wanted_columns],lomb_df,path=info_path)
        
                     
    pass

## Main Function

In [ ]:
def main():

    # Creating New Folder for phase curve plots
    outdir = os.getcwd()
    phase_path = os.path.join(outdir,"auto_plots")
    
    if not os.path.isdir(phase_path):
        os.mkdir(phase_path)

    mask = Ex_diaObj_df["comb_auto_filter"]

    lomb_df = pd.read_csv("data/lomb_diaObj_ps.csv")

    Ex_diaObj_df[mask].apply(filter_table,lomb_df=lomb_df,path=phase_path,axis=1)


In [ ]:
main()